In [1]:
%%writefile data.csv
movie_id,title,genre,budget,revenue,release_date
1,Skyfall,Action,150,700,2012-11-09
2,Frozen,Animation,120,400,2013-11-27
3,Joker,Thriller,60,1000,2019-10-04
4,Parasite,Drama,11,300,2019-05-30
5,Inception,Sci-Fi,160,830,2010-07-16
6,Interstellar,Sci-Fi,165,650,2014-11-07
7,Avengers: Endgame,Action,356,2797,2019-04-26
8,Inside Out,Animation,175,857,2015-06-19
9,Titanic,Drama,200,2200,1997-12-19
10,The Dark Knight,Action,185,1005,2008-07-18
11,Up,Animation,175,735,2009-05-29
12,Gravity,Sci-Fi,100,723,2013-10-04
13,Get Out,Thriller,4.5,255,2017-02-24
14,Black Panther,Action,200,1347,2018-02-16
15,La La Land,Drama,30,446,2016-12-09
16,Minions,Animation,74,1159,2015-07-10
17,Avatar,Sci-Fi,237,2923,2009-12-18
18,The Martian,Sci-Fi,108,630,2015-10-02
19,Shutter Island,Thriller,80,295,2010-02-19
20,1917,Drama,95,384,2019-12-25

Writing data.csv


In [2]:
#Create SparkSession & load dataframe

#Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark= SparkSession.builder.getOrCreate()

df = spark.read.csv("data.csv", header=True, inferSchema=True)
df.show()

df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/07 06:29:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/07 06:29:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+--------+-----------------+---------+------+-------+------------+
|movie_id|            title|    genre|budget|revenue|release_date|
+--------+-----------------+---------+------+-------+------------+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04|
|       4|         Parasite|    Drama|  11.0|    300|  2019-05-30|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-07|
|       7|Avengers: Endgame|   Action| 356.0|   2797|  2019-04-26|
|       8|       Inside Out|Animation| 175.0|    857|  2015-06-19|
|       9|          Titanic|    Drama| 200.0|   2200|  1997-12-19|
|      10|  The Dark Knight|   Action| 185.0|   1005|  2008-07-18|
|      11|               Up|Animation| 175.0|    735|  2009-05-29|
|      12|          Gravity|   Sci-Fi| 100.0|    723|  2013-10

In [10]:
#Add Year Column (extract from date)
df=df.withColumn("Year",F.year( F.to_date("release_date")))
df.show()

+--------+-----------------+---------+------+-------+------------+------+----+
|movie_id|            title|    genre|budget|revenue|release_date|Profit|Year|
+--------+-----------------+---------+------+-------+------------+------+----+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09| 550.0|2012|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27| 280.0|2013|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04| 940.0|2019|
|       4|         Parasite|    Drama|  11.0|    300|  2019-05-30| 289.0|2019|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16| 670.0|2010|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-07| 485.0|2014|
|       7|Avengers: Endgame|   Action| 356.0|   2797|  2019-04-26|2441.0|2019|
|       8|       Inside Out|Animation| 175.0|    857|  2015-06-19| 682.0|2015|
|       9|          Titanic|    Drama| 200.0|   2200|  1997-12-19|2000.0|1997|
|      10|  The Dark Knight|   Action| 185.0|   1005

In [11]:
#Calculate Profit (revenue - budget)
df=df.withColumn("Profit", F.col("revenue")-F.col("budget"))
df.show()

+--------+-----------------+---------+------+-------+------------+------+----+
|movie_id|            title|    genre|budget|revenue|release_date|Profit|Year|
+--------+-----------------+---------+------+-------+------------+------+----+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09| 550.0|2012|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27| 280.0|2013|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04| 940.0|2019|
|       4|         Parasite|    Drama|  11.0|    300|  2019-05-30| 289.0|2019|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16| 670.0|2010|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-07| 485.0|2014|
|       7|Avengers: Endgame|   Action| 356.0|   2797|  2019-04-26|2441.0|2019|
|       8|       Inside Out|Animation| 175.0|    857|  2015-06-19| 682.0|2015|
|       9|          Titanic|    Drama| 200.0|   2200|  1997-12-19|2000.0|1997|
|      10|  The Dark Knight|   Action| 185.0|   1005

In [13]:
#Add Profit Margin (profti/budger)x100
df=df.withColumn("Profit_Margin", F.round(F.col("profit")/F.col("budget")*100,2))
df.show()

+--------+-----------------+---------+------+-------+------------+------+----+-------------+
|movie_id|            title|    genre|budget|revenue|release_date|Profit|Year|Profit_Margin|
+--------+-----------------+---------+------+-------+------------+------+----+-------------+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09| 550.0|2012|       366.67|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27| 280.0|2013|       233.33|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04| 940.0|2019|      1566.67|
|       4|         Parasite|    Drama|  11.0|    300|  2019-05-30| 289.0|2019|      2627.27|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16| 670.0|2010|       418.75|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-07| 485.0|2014|       293.94|
|       7|Avengers: Endgame|   Action| 356.0|   2797|  2019-04-26|2441.0|2019|       685.67|
|       8|       Inside Out|Animation| 175.0|    857|  2015-06-19| 682

In [14]:
#Categorize revenue
#Values - [0-200 'Low' | 200-500 'Good' | 500-700 'High' | 700 above - 'Blockbuster']
df=df.withColumn("revenue_category", F.when(
    F.col("revenue").between(0,199),"Low")
    .when(F.col("revenue").between(200,499),"Good")
    .when(F.col("revenue").between(500,699),"High")
    .when(F.col("revenue")>=700,"Blockbuster"))
df.show()

+--------+-----------------+---------+------+-------+------------+------+----+-------------+----------------+
|movie_id|            title|    genre|budget|revenue|release_date|Profit|Year|Profit_Margin|revenue_category|
+--------+-----------------+---------+------+-------+------------+------+----+-------------+----------------+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09| 550.0|2012|       366.67|     Blockbuster|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27| 280.0|2013|       233.33|            Good|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04| 940.0|2019|      1566.67|     Blockbuster|
|       4|         Parasite|    Drama|  11.0|    300|  2019-05-30| 289.0|2019|      2627.27|            Good|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16| 670.0|2010|       418.75|     Blockbuster|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-07| 485.0|2014|       293.94|            High|
|       7|

In [15]:
#Filter profitable (>300)
df.filter(F.col("revenue")>300).show()

+--------+-----------------+---------+------+-------+------------+------+----+-------------+----------------+
|movie_id|            title|    genre|budget|revenue|release_date|Profit|Year|Profit_Margin|revenue_category|
+--------+-----------------+---------+------+-------+------------+------+----+-------------+----------------+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09| 550.0|2012|       366.67|     Blockbuster|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27| 280.0|2013|       233.33|            Good|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04| 940.0|2019|      1566.67|     Blockbuster|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16| 670.0|2010|       418.75|     Blockbuster|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-07| 485.0|2014|       293.94|            High|
|       7|Avengers: Endgame|   Action| 356.0|   2797|  2019-04-26|2441.0|2019|       685.67|     Blockbuster|
|       8|

In [17]:
#Genre wise summary (Agg total revenue, budget & profit margin by genre)
df.groupby("genre").agg(
    F.sum("revenue").alias("total_revenue"),
    F.sum("budget").alias("total_budget"),
    F.sum("Profit").alias("profit_margin")
).show()

+---------+-------------+------------+-------------+
|    genre|total_revenue|total_budget|profit_margin|
+---------+-------------+------------+-------------+
| Thriller|         1550|       144.5|       1405.5|
|    Drama|         3330|       336.0|       2994.0|
|Animation|         3151|       544.0|       2607.0|
|   Action|         5849|       891.0|       4958.0|
|   Sci-Fi|         5756|       770.0|       4986.0|
+---------+-------------+------------+-------------+



In [19]:
#Add total column (add net total i.e. revenue - budget using "expr")
df.withColumn("Total", F.expr("revenue - budget")).show()

+--------+-----------------+---------+------+-------+------------+------+----+-------------+----------------+------+
|movie_id|            title|    genre|budget|revenue|release_date|Profit|Year|Profit_Margin|revenue_category| Total|
+--------+-----------------+---------+------+-------+------------+------+----+-------------+----------------+------+
|       1|          Skyfall|   Action| 150.0|    700|  2012-11-09| 550.0|2012|       366.67|     Blockbuster| 550.0|
|       2|           Frozen|Animation| 120.0|    400|  2013-11-27| 280.0|2013|       233.33|            Good| 280.0|
|       3|            Joker| Thriller|  60.0|   1000|  2019-10-04| 940.0|2019|      1566.67|     Blockbuster| 940.0|
|       4|         Parasite|    Drama|  11.0|    300|  2019-05-30| 289.0|2019|      2627.27|            Good| 289.0|
|       5|        Inception|   Sci-Fi| 160.0|    830|  2010-07-16| 670.0|2010|       418.75|     Blockbuster| 670.0|
|       6|     Interstellar|   Sci-Fi| 165.0|    650|  2014-11-0

In [28]:
#Print list of top 2 genres based on revenue (use flatMap)
dd=df.groupby("genre").agg(F.sum("revenue").alias("revenue_per_genre")).orderBy(F.col("revenue_per_genre").desc()).select("genre").limit(2).rdd.flatMap(lambda x:x).collect()

dd

['Action', 'Sci-Fi']